# Step 1: Local Validation Strategy

## Why do we need this?
You noticed that your "Trend" submission got a *worse* score (28,068) than your "Weekday Mean" submission (23,335). 

To avoid guessing, we need a way to test our models **locally** (on your computer) before uploading to Kaggle. This saves time and submissions.

## The Strategy
We will hide the last 28 days of data from our model. 
- **Training Set**: All data *except* the last 28 days.
- **Validation Set**: The last 28 days.

We will train on the Training Set and predict the Validation Set. Since we know the *real* answers for the Validation Set, we can calculate the error (RMSE) ourselves!

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

# 1. Load Data
train = pd.read_csv("../data/train.csv")
train["date"] = pd.to_datetime(train["date"])

# 2. Create the Split
last_date = train["date"].max()
cutoff_date = last_date - pd.Timedelta(days=28)

print(f"Training Data Ends: {cutoff_date}")
print(f"Validation Data Starts: {cutoff_date + pd.Timedelta(days=1)}")

train_set = train[train["date"] <= cutoff_date].copy()
val_set = train[train["date"] > cutoff_date].copy()

print(f"Train shape: {train_set.shape}")
print(f"Val shape: {val_set.shape}")

Training Data Ends: 2015-09-02 00:00:00
Validation Data Starts: 2015-09-03 00:00:00
Train shape: (18458, 4)
Val shape: (308, 4)


## Experiment 1: Weekday Mean Model
Let's test the model that gave you the best score (23,335).

In [2]:
# Calculate average sales per store per weekday using ONLY the training set
train_set["weekday"] = train_set["date"].dt.weekday
val_set["weekday"] = val_set["date"].dt.weekday

store_weekday_mean = train_set.groupby(["store_id", "weekday"])["revenue"].mean()

# Predict on Validation Set
val_set["pred_weekday_mean"] = val_set.set_index(["store_id", "weekday"]).index.map(store_weekday_mean)

# Fill missing values with global mean (if any)
val_set["pred_weekday_mean"] = val_set["pred_weekday_mean"].fillna(train_set["revenue"].mean())

# Calculate RMSE
rmse_weekday = np.sqrt(mean_squared_error(val_set["revenue"], val_set["pred_weekday_mean"]))
print(f"Local Validation RMSE (Weekday Mean): {rmse_weekday:.2f}")

Local Validation RMSE (Weekday Mean): 21262.81


## Experiment 2: Linear Trend Model
Let's test the model that gave you the worse score (28,068). Why did it fail?

In [3]:
from sklearn.linear_model import LinearRegression

trend_preds = []

for store_id, df in train_set.groupby("store_id"):
    # Take last 28 days of TRAINING data to calculate trend
    recent_df = df.sort_values("date").tail(28)
    
    X = np.arange(len(recent_df)).reshape(-1, 1)
    y = recent_df["revenue"].values
    
    model = LinearRegression()
    model.fit(X, y)
    
    # Predict for the NEXT 28 days (which is our validation period)
    # The validation days are 28, 29, ..., 55 days ahead relative to the start of the 'recent_df' window
    # Actually, simpler: we just need to predict 28 steps into the future from the end of training
    future_X = np.arange(len(recent_df), len(recent_df) + 28).reshape(-1, 1)
    pred = model.predict(future_X)
    
    # Store predictions
    temp_df = pd.DataFrame({
        "store_id": store_id,
        "date": val_set[val_set["store_id"] == store_id].sort_values("date")["date"].values,
        "pred_trend": pred
    })
    trend_preds.append(temp_df)

trend_df = pd.concat(trend_preds)

# Merge back to validation set
val_set = val_set.merge(trend_df, on=["store_id", "date"], how="left")

# Calculate RMSE
rmse_trend = np.sqrt(mean_squared_error(val_set["revenue"], val_set["pred_trend"]))
print(f"Local Validation RMSE (Linear Trend): {rmse_trend:.2f}")

Local Validation RMSE (Linear Trend): 19245.78


## Conclusion
Compare the two RMSE numbers above. 
- If `rmse_trend` > `rmse_weekday`, it confirms locally what you saw on the leaderboard!
- This proves that simply drawing a straight line (trend) is dangerous because sales fluctuate wildly (weekends vs weekdays).

**Next Step**: We will build a model that uses BOTH (and more) using LightGBM.